# 01 · Build a TLE file from satellite names

Give it satellite names (or NORAD numbers). It finds their element sets and writes one 3-line TLE file that
`02_overflight_india.ipynb` reads.

Two sources:

- **`"celestrak"`** (default): CelesTrak's public GP service, which needs no account or credentials. It returns
  each satellite's **latest** element set, which is right for recent dates (within a few days).
- **`"local"`**: search a file you downloaded yourself, for example a Space-Track 3LE catalogue or daily elset file.
  Use this for older dates; every element set found for a satellite is kept, and notebook 02 picks the one
  nearest each time window.

Names are matched leniently: case, spacing and leading zeros are ignored, so `IONOSFERA-M 1` matches Space-Track's
`IONOSFERA-M 01`. If a name matches several objects, they are listed and nothing is guessed.

In [5]:
import re
import time
import urllib.parse
import urllib.request
from datetime import datetime, timedelta, timezone

import pandas as pd

# ---- inputs ------------------------------------------------------------------
SATELLITE_NAMES = [           # names as they appear in the catalogue, or NORAD numbers as strings
    "IONOSFERA-M 1",
    "IONOSFERA-M 2",
    "IONOSFERA-M 3",
    "IONOSFERA-M 4",
]
SOURCE = "celestrak"          # "celestrak" or "local"
LOCAL_FILE = "tle/catalogue.tle"   # used when SOURCE == "local"
CELESTRAK_URL = "https://celestrak.org/NORAD/elements/gp.php"
OUT_FILE = "tle/satellites.tle"   # notebook 02 reads this path by default

In [6]:
def normalise(name):
    """Upper case, no spaces or underscores, unified dashes, no leading zeros in numbers."""
    s = name.upper().replace("_", " ").replace("–", "-").replace("—", "-")
    s = re.sub(r"(?<!\d)0+(?=\d)", "", s)
    return re.sub(r"\s+", "", s)


def checksum_ok(line):
    if len(line) < 69 or not line[68].isdigit():
        return False
    return sum(int(c) if c.isdigit() else (1 if c == "-" else 0) for c in line[:68]) % 10 == int(line[68])


def epoch_of(line1):
    yy, doy = int(line1[18:20]), float(line1[20:32])
    return datetime(2000 + yy if yy < 57 else 1900 + yy, 1, 1, tzinfo=timezone.utc) + timedelta(days=doy - 1)


def parse_tle_text(text):
    """[(name, line1, line2)] from 2- or 3-line text; bad checksums are skipped."""
    lines = [ln.rstrip() for ln in text.splitlines()]
    out, bad = [], 0
    for i in range(len(lines) - 1):
        l1, l2 = lines[i], lines[i + 1]
        if not (l1.startswith("1 ") and l2.startswith("2 ")):
            continue
        if not (checksum_ok(l1) and checksum_ok(l2)) or l1[2:7] != l2[2:7]:
            bad += 1
            continue
        prev = lines[i - 1].strip() if i > 0 else ""
        name = prev[2:].strip() if prev.startswith("0 ") else (prev if prev and not prev[:2] in ("1 ", "2 ") else l1[2:7].strip())
        out.append((name, l1, l2))
    if bad:
        print(f"  skipped {bad} element set(s) with a bad checksum")
    return out


def pick(query, records):
    """Records for one query: NORAD number, else exact normalised name, else a single partial match."""
    q = query.strip()
    if q.isdigit():
        return [r for r in records if int(r[1][2:7]) == int(q)], []
    exact = [r for r in records if normalise(r[0]) == normalise(q)]
    if exact:
        return exact, []
    partial = [r for r in records if normalise(q) in normalise(r[0])]
    names = sorted({r[0] for r in partial})
    return (partial, []) if len(names) == 1 else ([], names)

In [7]:
_cache = {}


def _get(key, value, attempts=4):
    """One CelesTrak query, retried with a growing pause: a single timeout should not lose a satellite."""
    if (key, value) not in _cache:
        url = f"{CELESTRAK_URL}?{key}={urllib.parse.quote(value)}&FORMAT=TLE"
        req = urllib.request.Request(url, headers={"User-Agent": "asteria-notebook/1.0"})
        for attempt in range(1, attempts + 1):
            try:
                with urllib.request.urlopen(req, timeout=30) as resp:
                    _cache[(key, value)] = parse_tle_text(resp.read().decode("utf-8", errors="replace"))
                break
            except OSError as exc:            # timeouts, refused or reset connections, DNS failures
                if attempt == attempts:
                    raise
                wait = 5 * attempt
                print(f"  {key}={value}: attempt {attempt} failed ({exc}); retrying in {wait} s")
                time.sleep(wait)
        time.sleep(1)             # be gentle with a free service
    return _cache[(key, value)]


def from_celestrak(query):
    """CelesTrak's NAME search is a plain substring match, so 'IONOSFERA-M 1' would miss 'IONOSFERA-M 01'.
    Search the name without its trailing number, then match locally with normalise()."""
    q = query.strip()
    if q.isdigit():
        return _get("CATNR", q)
    stem = re.sub(r"[\s\-_]*\d+$", "", q) or q
    return _get("NAME", stem) or _get("NAME", q)


local_records = None
if SOURCE == "local":
    with open(LOCAL_FILE, encoding="utf-8", errors="replace") as fh:
        local_records = parse_tle_text(fh.read())
    print(f"{len(local_records):,} element sets in {LOCAL_FILE}")

found, rows = [], []
for query in SATELLITE_NAMES:
    try:
        records = local_records if SOURCE == "local" else from_celestrak(query)
    except Exception as exc:
        rows.append({"query": query, "status": f"fetch failed: {exc}"})
        continue
    hits, ambiguous = pick(query, records)
    if ambiguous:
        rows.append({"query": query, "status": "ambiguous: " + "; ".join(ambiguous[:10])})
        continue
    if not hits:
        rows.append({"query": query, "status": "not found"})
        continue
    for name, l1, l2 in hits:
        ep = epoch_of(l1)
        found.append((query, name, l1, l2, ep))
        rows.append({"query": query, "status": "ok", "catalogue name": name, "NORAD": int(l1[2:7]),
                     "epoch (UTC)": f"{ep:%Y-%m-%d %H:%M:%S}",
                     "age (days)": round((datetime.now(timezone.utc) - ep).total_seconds() / 86400, 2)})
report = pd.DataFrame(rows)
report

,query,status,catalogue name,NORAD,epoch (UTC),age (days)
0,IONOSFERA-M 1,ok,IONOSFERA-M 1,61735,2026-10-08 06:39:08,0.28
1,IONOSFERA-M 2,ok,IONOSFERA-M 2,61736,2026-10-08 05:46:52,0.32
2,IONOSFERA-M 3,ok,IONOSFERA-M 3,64876,2026-10-08 06:56:49,0.27
3,IONOSFERA-M 4,ok,IONOSFERA-M 4,64877,2026-10-08 06:05:21,0.30


In [8]:
import os

# de-duplicate identical element sets, sort by satellite then epoch, write 3-line format
seen, out_lines = set(), []
for query, name, l1, l2, ep in sorted(found, key=lambda r: (int(r[2][2:7]), r[4])):
    if (l1, l2) in seen:
        continue
    seen.add((l1, l2))
    out_lines += [f"0 {query.strip() if not query.strip().isdigit() else name}", l1, l2]

if out_lines:
    os.makedirs(os.path.dirname(OUT_FILE) or ".", exist_ok=True)
    with open(OUT_FILE, "w") as fh:
        fh.write("\n".join(out_lines) + "\n")
    print(f"Wrote {len(out_lines) // 3} element set(s) to {OUT_FILE}")
else:
    print("Nothing found; no file written.")
missing = report[report.status != "ok"]
if len(missing):
    print("Check these:", ", ".join(missing["query"]))

Wrote 4 element set(s) to tle/satellites.tle
